In [ ]:
# =============================================================
#  BANK MARKETING PROJECT (Machine Learning)
# =============================================================

In [ ]:
# ==============================
# 1. IMPORTS
# ==============================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE, ADASYN

In [ ]:
# ==============================
# 2. LOAD DATASET
# ==============================
df = pd.read_csv('/content/Bank_Marketing_Full.csv', sep=';')

In [8]:
# ==============================
# 3. PREPROCESSING
# ==============================
# Drop Leakage Feature
df = df.drop('duration', axis=1)

# Handle "unknown" Values
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].replace('unknown', df[col].mode()[0])

# Encode Target
df['y'] = df['y'].map({'no': 0, 'yes': 1})

# One-Hot Encoding
df = pd.get_dummies(df, drop_first=True)


# ==============================
# 4. SPLIT DATA
# ==============================
# Split Features & Target
X = df.drop('y', axis=1)
y = df['y']

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Sampling Techniques
smote = SMOTE(random_state=42)
adasyn = ADASYN(random_state=42)

X_train_sm, y_train_sm = smote.fit_resample(X_train_scaled, y_train)
X_train_ad, y_train_ad = adasyn.fit_resample(X_train_scaled, y_train)


# ==============================
# 5. MODELS
# ==============================
models = {
    "Decision Tree": DecisionTreeClassifier(
        max_depth=10,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42
    ),

    "MLP": MLPClassifier(
        hidden_layer_sizes=(128, 64),
        max_iter=500,
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        eval_metric='logloss',
        random_state=42
    )
}


# ==============================
# 6. METRICS
# ==============================
scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}


# ==============================
# 7. CROSS VALIDATION
# ==============================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


# ==============================
# 8. EVALUATION FUNCTION
# ==============================
def evaluate_models(X, y, label):
    print("\n" + "="*30)
    print(f"  {label}")
    print("="*30)

    for name, model in models.items():
        scores = cross_validate(
            model,
            X,
            y,
            cv=skf,
            scoring=scoring,
            n_jobs=-1
        )

        print(f"\nModel: {name}")
        print(f"Accuracy : {np.mean(scores['test_accuracy']):.4f}")
        print(f"Precision: {np.mean(scores['test_precision']):.4f}")
        print(f"Recall   : {np.mean(scores['test_recall']):.4f}")
        print(f"F1 Score : {np.mean(scores['test_f1']):.4f}")
        print(f"ROC-AUC  : {np.mean(scores['test_roc_auc']):.4f}")


# ==============================
# 9. RUN EXPERIMENTS
# ==============================

# Baseline
evaluate_models(X_train_scaled, y_train, "Baseline (No Sampling)")

# SMOTE
evaluate_models(X_train_sm, y_train_sm, "SMOTE")

# ADASYN
evaluate_models(X_train_ad, y_train_ad, "ADASYN")


  Baseline (No Sampling)

Model: Decision Tree
Accuracy : 0.8948
Precision: 0.5695
Recall   : 0.2721
F1 Score : 0.3679
ROC-AUC  : 0.7302

Model: Random Forest
Accuracy : 0.8929
Precision: 0.5489
Recall   : 0.2823
F1 Score : 0.3726
ROC-AUC  : 0.7715

Model: MLP
Accuracy : 0.8568
Precision: 0.3493
Recall   : 0.3128
F1 Score : 0.3294
ROC-AUC  : 0.6718

Model: XGBoost
Accuracy : 0.8979
Precision: 0.6095
Recall   : 0.2619
F1 Score : 0.3661
ROC-AUC  : 0.7899

  SMOTE

Model: Decision Tree
Accuracy : 0.8747
Precision: 0.9230
Recall   : 0.8176
F1 Score : 0.8671
ROC-AUC  : 0.9220

Model: Random Forest
Accuracy : 0.9392
Precision: 0.9467
Recall   : 0.9307
F1 Score : 0.9386
ROC-AUC  : 0.9821

Model: MLP
Accuracy : 0.9163
Precision: 0.9063
Recall   : 0.9288
F1 Score : 0.9173
ROC-AUC  : 0.9673

Model: XGBoost
Accuracy : 0.9362
Precision: 0.9683
Recall   : 0.9020
F1 Score : 0.9340
ROC-AUC  : 0.9738

  ADASYN

Model: Decision Tree
Accuracy : 0.8701
Precision: 0.9239
Recall   : 0.8063
F1 Score : 0.86